# BƯỚC 9 — Bổ sung tiêu chí 7: Baseline, khoảng dao động, sai số và chi phí giả định
**Đề tài 5: Bệnh viện — Dự đoán tái nhập viện trong 30 ngày.**

Notebook này **không huấn luyện**, **không tinh chỉnh ngưỡng** và **không chọn lại mô hình** bằng Test. Quyết định từ Validation phải được khóa là `DecisionTree`, ngân sách theo dõi là **10% lượt xuất viện**.

Các phân tích gồm:
1. Đối chiếu Baseline và các chỉ số phân lớp trên Test với Bước 8.
2. Ước lượng **percentile bootstrap 95% theo cụm bệnh nhân** cho các chỉ số của mô hình đã khóa; dùng cùng mẫu bootstrap khi so sánh hai mô hình.
3. Phân tích **đồng hạng điểm nguy cơ** tại ranh giới Top 10% của DecisionTree.
4. Phân tích **chi phí sai lầm theo kịch bản giả định** (không phải chi phí y tế thực tế).

Các output chỉ là **số liệu tổng hợp**, không xuất danh sách bệnh nhân/encounter. Giữ tất cả dữ liệu và việc chạy notebook **trên máy cục bộ**.

## 1. Khai báo thư viện, đường dẫn và quy tắc khóa mô hình
Notebook dùng dữ liệu `data/processed/step4/` và mô hình `reports/classification/`. Chạy JupyterLab trong `D:\Readmission_Project` hoặc từ thư mục `notebooks`.

**Lưu ý bảo mật:** File `.joblib` dùng định dạng pickle. Chỉ mở file mô hình do nhóm tự tạo và tin cậy. Nên dùng cùng phiên bản scikit-learn với lúc huấn luyện.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score,
)
from IPython.display import display

SEED = 42
TOP_FRACTION = 0.10
BOOTSTRAP_REPS = 1000
TIE_SIMULATIONS = 1000
MODEL_NAMES = ['Baseline_majority', 'DecisionTree', 'RandomForest']

cwd = Path.cwd()
ROOT = next((p for p in [cwd, *cwd.parents] if
             (p / 'data' / 'processed' / 'step4' / 'X_test.npz').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Mo JupyterLab tu D:/Readmission_Project hoac thu muc notebooks')
DATA = ROOT / 'data' / 'processed' / 'step4'
CLASS_REPORT = ROOT / 'reports' / 'classification'
EVAL_REPORT = ROOT / 'reports' / 'evaluation_fairness'
OUT = ROOT / 'reports' / 'model_stability'
OUT.mkdir(parents=True, exist_ok=True)

locked = json.loads((CLASS_REPORT / 'training_summary.json').read_text(encoding='utf-8'))
assert locked['selected_on_validation'] == 'DecisionTree', 'LUA CHON VALIDATION KHAC DECISION TREE!'
assert locked['test_used'] is False, 'Test da su dung trong lua chon model!'
assert abs(float(locked['top_fraction']) - TOP_FRACTION) < 1e-12
assert locked['random_state'] == SEED
print('Nguon:', ROOT)
print('Model khoa:', locked['selected_on_validation'])
print('Top fraction:', TOP_FRACTION)

## 2. Nạp và kiểm tra dữ liệu Test
Ranh giới Train/Validation/Test phải tách theo `patient_nbr`, tránh cho cùng người xuất hiện ở nhiều tập. Kiểm tra bằng metadata ID, không dùng ID làm đặc trưng dự báo.

In [ ]:
X = sparse.load_npz(DATA / 'X_test.npz').tocsr()
y = np.load(DATA / 'y_test.npy').astype(np.int64)
test_ids = pd.read_csv(DATA / 'test_ids_labels.csv')
train_ids = pd.read_csv(DATA / 'train_ids_labels.csv')
val_ids = pd.read_csv(DATA / 'validation_ids_labels.csv')
assert len(y) == X.shape[0] == len(test_ids) == 15219
assert X.shape[1] == locked['n_features']
assert test_ids.encounter_id.is_unique
assert np.array_equal(y, test_ids.readmitted_30d.to_numpy(dtype=np.int64))
assert set(test_ids.patient_nbr).isdisjoint(train_ids.patient_nbr)
assert set(test_ids.patient_nbr).isdisjoint(val_ids.patient_nbr)
assert set(train_ids.patient_nbr).isdisjoint(val_ids.patient_nbr)
assert np.array_equal(np.unique(y), [0, 1])
assert int(y.sum()) == 1680
n = len(y)
k = int(np.ceil(TOP_FRACTION * n))
patient_codes, unique_patients = pd.factorize(test_ids.patient_nbr, sort=False)
num_patients = len(unique_patients)
print('Test:', X.shape, '| Positive:', int(y.sum()), '| Patients:', num_patients, '| Calls:', k)

## 3. Baseline và các chỉ số tại ngưỡng 0,5
**Baseline majority** dự đoán mọi lượt thuộc lớp 0. Accuracy đơn độc sẽ gây hiểu lầm vì nhãn dương chỉ chiếm ~11%. Dùng Precision, Recall, F1, Average Precision (AP), ROC-AUC và ma trận nhầm lẫn.

Bảng được **đối chiếu với file Bước 8** để đảm bảo không có việc thay đổi mô hình hoặc dữ liệu. Nếu sai lệch, notebook dừng và yêu cầu kiểm tra môi trường/phiên bản scikit-learn.

In [ ]:
models = {name: joblib.load(CLASS_REPORT / f'model_{name}.joblib')
          for name in MODEL_NAMES}
score = {name: models[name].predict_proba(X)[:, list(models[name].classes_).index(1)]
         for name in MODEL_NAMES}
metrics_rows = []
for name in MODEL_NAMES:
    yp = (score[name] >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    metrics_rows.append({
        'model': name, 'n_test': n, 'threshold': 0.5,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'precision': precision_score(y, yp, zero_division=0),
        'recall': recall_score(y, yp, zero_division=0),
        'f1': f1_score(y, yp, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y, yp),
        'average_precision': average_precision_score(y, score[name]),
        'roc_auc': roc_auc_score(y, score[name]),
    })
threshold_metrics = pd.DataFrame(metrics_rows)
previous = pd.read_csv(EVAL_REPORT / 'test_metrics_threshold_050.csv')
for name in MODEL_NAMES:
    new = threshold_metrics.set_index('model').loc[name]
    old = previous.set_index('model').loc[name]
    for col in ['TP','FP','TN','FN']:
        assert int(new[col]) == int(old[col]), (name, col, new[col], old[col])
    for col in ['precision','recall','f1','average_precision','roc_auc']:
        assert abs(float(new[col])-float(old[col])) < 0.00015, (name,col)
threshold_metrics.to_csv(OUT / 'threshold_metrics_verified.csv', index=False, encoding='utf-8-sig')
display(threshold_metrics.round(4))
print('PASS: Metrics va confusion matrix khop output Buoc 8.')

## 4. Recall@10%, Precision@10%, Lift@10%
Định nghĩa Top 10% giữ nguyên cách tính Bước 8: `np.argsort(-scores, kind='stable')` rồi chọn `ceil(0.1*n)` lượt.

**Không coi Baseline majority là thuật toán xếp hạng**: tất cả điểm dự đoán bằng nhau. Cột Baseline từ Bước 8 chỉ là thứ tự phá hòa của dữ liệu. Benchmark chính để diễn giải là **lựa chọn ngẫu nhiên theo năng lực 10%**, kỳ vọng Recall = `k/n` và Precision = tỷ lệ dương.

In [ ]:
def rank_top_metrics(labels, scores, fraction=TOP_FRACTION):
    count = len(labels)
    budget = max(1, int(np.ceil(fraction * count)))
    order = np.argsort(-scores, kind='stable')
    selected = order[:budget]
    positives = int(np.sum(labels, dtype=np.int64))
    hits = int(np.sum(labels[selected], dtype=np.int64))
    precision = hits / budget
    recall = hits / positives if positives else np.nan
    prevalence = positives / count
    return {'n': count, 'calls': budget, 'positives': positives,
            'hits': hits, 'precision_at_10pct': precision,
            'recall_at_10pct': recall,
            'lift_at_10pct': precision / prevalence if prevalence else np.nan}

rows = []
for name in MODEL_NAMES:
    row = rank_top_metrics(y, score[name]); row['model'] = name
    rows.append(row)
top_metrics = pd.DataFrame(rows)
old_top = pd.read_csv(EVAL_REPORT / 'test_capacity_top10pct.csv').set_index('model')
for row in rows:
    assert row['hits'] == int(old_top.loc[row['model'], 'true_readmissions_captured'])
top_metrics.to_csv(OUT / 'top10_metrics_verified.csv', index=False, encoding='utf-8-sig')
display(top_metrics.round(4))
print('Random expectation: Recall@10% =', round(k/n,4),
      '| expected hits =', round(k * y.mean(),1))
print('PASS: Top-10 hits khop Buoc 8.')

## 5. Độ nhạy với cách phá hòa điểm (tie-breaking)
DecisionTree có thể trả cùng một xác suất cho rất nhiều lượt. Nếu Top 10% cắt giữa một nhóm đồng điểm, kết quả phụ thuộc vào thứ tự các dòng khi dùng `stable`. Điều này cần được minh bạch.

Thử nghiệm sau xáo trộn **ngẫu nhiên trong các lượt có cùng điểm**, không dùng nhãn để ưu tiên. Đây là **phân tích độ nhạy vận hành**, KHÔNG phải khoảng tin cậy thống kê. Không dùng thử nghiệm này để chọn lại mô hình/siêu tham số.

In [ ]:
tie_rows = []
for name in ['DecisionTree','RandomForest']:
    sc = score[name]
    order = np.argsort(-sc, kind='stable')
    cutoff_score = sc[order[k-1]]
    above = sc > cutoff_score
    at_cutoff = sc == cutoff_score
    slots_left = k - int(above.sum())
    tie_rows.append({
        'model': name, 'cutoff_score': float(cutoff_score),
        'n_above_cutoff': int(above.sum()),
        'n_equal_cutoff': int(at_cutoff.sum()),
        'slots_from_tie': slots_left,
        'positives_in_tie': int(np.sum(y[at_cutoff], dtype=np.int64)),
        'positives_above': int(np.sum(y[above], dtype=np.int64)),
    })
tie_boundary = pd.DataFrame(tie_rows)
tie_boundary.to_csv(OUT / 'tie_boundary_aggregate.csv', index=False, encoding='utf-8-sig')
display(tie_boundary)

rng_ties = np.random.default_rng(SEED + 200)
sc = score['DecisionTree']
base_order = np.argsort(-sc, kind='stable')
cutoff_score = sc[base_order[k-1]]
above_idx = np.flatnonzero(sc > cutoff_score)
tied_idx = np.flatnonzero(sc == cutoff_score)
slots = k - len(above_idx)
above_hits = int(np.sum(y[above_idx], dtype=np.int64))
tie_hits = np.empty(TIE_SIMULATIONS, dtype=np.int64)
for i in range(TIE_SIMULATIONS):
    chosen = rng_ties.choice(tied_idx, size=slots, replace=False)
    tie_hits[i] = above_hits + int(np.sum(y[chosen], dtype=np.int64))
tie_stats = pd.DataFrame([{
    'runs': TIE_SIMULATIONS,
    'observed_hits_stable': int(top_metrics.set_index('model').loc['DecisionTree','hits']),
    'random_tie_mean_hits': float(tie_hits.mean()),
    'random_tie_q025_hits': float(np.quantile(tie_hits,.025)),
    'random_tie_q975_hits': float(np.quantile(tie_hits,.975)),
    'random_tie_min_hits': int(tie_hits.min()),
    'random_tie_max_hits': int(tie_hits.max()),
}])
tie_stats.to_csv(OUT / 'tie_sensitivity_summary.csv', index=False, encoding='utf-8-sig')
display(tie_stats.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(tie_hits, bins=max(6, len(np.unique(tie_hits))))
ax.axvline(int(top_metrics.set_index('model').loc['DecisionTree','hits']), linestyle='--',
           label='Ket qua stable goc')
ax.set(xlabel='So ca phat hien trong Top 10%', ylabel='So lan mo phong',
       title='Do nhay khi pha hoa ngau nhien trong nhom dong diem')
ax.legend(); fig.tight_layout()
fig.savefig(OUT / '01_tie_sensitivity.png', dpi=160)
plt.show()

## 6. Bootstrap theo cụm bệnh nhân — khoảng dao động 95%
Mỗi lần bootstrap lấy mẫu **bệnh nhân có hoàn lại**, sau đó lấy **toàn bộ lượt nhập viện của bệnh nhân đó**. Do một số người có nhiều lượt, tổng số lượt mỗi mẫu bootstrap có thể thay đổi. Việc này phù hợp hơn bootstrap độc lập theo từng dòng khi một bệnh nhân xuất hiện nhiều lần.

- Lặp 1.000 lần, `seed=42`; hai mô hình dùng cùng các mẫu để so sánh ghép cặp.
- Giữ nguyên mô hình, **không fit lại** và không tối ưu ngưỡng trên Test.
- Tính percentile 2,5% và 97,5% cho Recall@10%, Precision@10%, Lift@10%, Average Precision và chênh lệch Recall@10% giữa DecisionTree/RandomForest.
- Khoảng này phản ánh **dao động do lấy mẫu bệnh nhân trên Test, có điều kiện trên mô hình đã fit**; không bao gồm độ dao động do huấn luyện lại, thay bệnh viện hoặc thay giai đoạn thời gian.
- Khi một cây cho nhiều điểm đồng hạng, tie-breaking ổn định theo thứ tự dòng gốc của bản sao bootstrap; phân tích tie-breaking được báo cáo riêng ở mục 5.

In [ ]:
from time import perf_counter
start = perf_counter()
rng = np.random.default_rng(SEED)
row_ids = np.arange(n)
recs=[]
for rep in range(BOOTSTRAP_REPS):
    # Mỗi bệnh nhân được chọn với hoàn lại; lặp lại toàn bộ các lượt thuộc bệnh nhân ấy.
    picked = rng.integers(0, num_patients, size=num_patients)
    counts = np.bincount(picked, minlength=num_patients)
    idx = np.repeat(row_ids, counts[patient_codes])
    labels = y[idx]
    if labels.sum() == 0 or labels.sum() == len(labels):
        continue
    sample_prevalence = labels.mean()
    r = {'bootstrap_rep': rep, 'n_encounters': len(idx),
         'sample_prevalence': sample_prevalence,
         'random_expected_recall_at_10pct': int(np.ceil(.1*len(idx)))/len(idx)}
    for name in ['DecisionTree','RandomForest']:
        s = score[name][idx]
        m = rank_top_metrics(labels, s)
        r[f'{name}_recall_at_10pct'] = m['recall_at_10pct']
        r[f'{name}_precision_at_10pct'] = m['precision_at_10pct']
        r[f'{name}_lift_at_10pct'] = m['lift_at_10pct']
        r[f'{name}_average_precision'] = average_precision_score(labels, s)
    r['diff_DT_minus_RF_recall_at_10pct'] = (
        r['DecisionTree_recall_at_10pct'] - r['RandomForest_recall_at_10pct'])
    recs.append(r)
bootstrap = pd.DataFrame(recs)
assert len(bootstrap) >= BOOTSTRAP_REPS * .99, 'Qua nhieu mau bootstrap mot lop'
bootstrap.to_csv(OUT / 'bootstrap_patient_replicates_aggregate.csv', index=False, encoding='utf-8-sig')
ci_rows=[]
for column in [c for c in bootstrap.columns if c not in ['bootstrap_rep','n_encounters']]:
    vals = bootstrap[column].to_numpy(dtype=float)
    ci_rows.append({'metric': column, 'mean_bootstrap':vals.mean(),
                    'q025':np.quantile(vals,.025),'q975':np.quantile(vals,.975)})
bootstrap_ci=pd.DataFrame(ci_rows)
bootstrap_ci.to_csv(OUT / 'bootstrap_patient_ci95.csv', index=False, encoding='utf-8-sig')
print(f'Bootstrap completed: {len(bootstrap)} replications in {perf_counter()-start:.1f}s')
display(bootstrap_ci.round(4))

## 7. Biểu đồ so sánh khoảng dao động Recall@10%
Đường tham chiếu 10% là **mức Recall kỳ vọng khi chọn ngẫu nhiên 10% lượt**, không phải một mô hình dự báo. Nếu khoảng của chênh lệch DecisionTree – RandomForest bao gồm 0, không có cơ sở kết luận mô hình này vượt trội ổn định chỉ từ đánh giá này.

In [ ]:
rows=[]
for name in ['DecisionTree','RandomForest']:
    key=f'{name}_recall_at_10pct'
    row=bootstrap_ci.set_index('metric').loc[key]
    observed=float(top_metrics.set_index('model').loc[name,'recall_at_10pct'])
    rows.append((name,observed,float(row.q025),float(row.q975)))
fig,ax=plt.subplots(figsize=(8,4))
x=np.arange(len(rows))
obs=np.array([r[1] for r in rows]); low=np.array([r[2] for r in rows]); high=np.array([r[3] for r in rows])
ax.errorbar(x, obs, yerr=np.vstack([obs-low,high-obs]), fmt='o', capsize=6)
ax.axhline(TOP_FRACTION,linestyle='--',label='Ngau nhien (xap xi 10%)')
ax.set_xticks(x, [r[0] for r in rows])
ax.set(ylabel='Recall@10%', title='Patient-cluster bootstrap 95% CI (model co dinh)')
ax.legend();fig.tight_layout();fig.savefig(OUT/'02_recall10_bootstrap_ci.png',dpi=160)
plt.show()

## 8. Chi phí sai lầm theo giả định — KHÔNG phải chi phí bệnh viện thực tế
Ba kịch bản minh họa tỷ lệ *chi phí bỏ sót một trường hợp tái nhập sớm* / *chi phí gọi theo dõi cho một trường hợp không tái nhập*: 2:1, 5:1, 10:1.

Công thức: `chi_phi = C_FN * FN + C_FP * FP`. Với chính sách Top 10%, mọi phương án đều sử dụng cùng năng lực gọi; **FP ở đây được hiểu là cuộc gọi không trúng nhãn dương**, không thể dùng thay thế chi phí nhân sự thực tế. Các hệ số chỉ là **đơn vị chi phí giả định**.

Phân biệt hai chính sách: (a) ngưỡng 0,5 có số lượt cảnh báo thay đổi giữa mô hình; (b) Top 10% cố định số lượt được liên hệ. Không dùng Test để tối ưu tỷ số chi phí hay chọn mô hình.

In [ ]:
cost_rows=[]
original_top=top_metrics.set_index('model')
for policy in ['threshold_0.5','top_10pct']:
    for fn_ratio in [2, 5, 10]:
        for name in MODEL_NAMES:
            if policy=='threshold_0.5':
                row=threshold_metrics.set_index('model').loc[name]
                tp, fp, fn, tn = [int(row[c]) for c in ['TP','FP','FN','TN']]
                n_calls=tp+fp
            else:
                if name=='Baseline_majority':
                    # Khong co thu hang: benchmark NGẪU NHIÊN KY VONG.
                    n_calls=k
                    tp=k * y.mean()
                    fp=k-tp
                    fn=float(y.sum())-tp
                    tn=(n-k)-fn
                else:
                    row=original_top.loc[name]
                    n_calls=int(row.calls)
                    tp=int(row.hits)
                    fp=n_calls-tp
                    fn=int(y.sum())-tp
                    tn=(n-n_calls)-fn
            cost_rows.append({
                'policy':policy,'model':name if policy=='threshold_0.5' or name!='Baseline_majority' else 'Random_expected',
                'C_FN_assumed':fn_ratio,'C_FP_assumed':1,
                'n_calls':n_calls,'TP':tp,'FP':fp,'FN':fn,'TN':tn,
                'assumed_total_loss':fn_ratio*fn+fp,
            })
costs=pd.DataFrame(cost_rows)
costs.to_csv(OUT/'cost_scenarios_illustrative.csv',index=False,encoding='utf-8-sig')
display(costs[(costs.policy=='top_10pct') & (costs.C_FN_assumed==5)].round(2))

## 9. Ghi tổng kết để đưa vào báo cáo và nhật ký quyết định
Tạo bản tổng hợp ngắn để trích dẫn chính xác, không tự tạo kết quả.

**Cách diễn giải:**
- Một kết quả `Recall@10% = 21,07%` nói về **tỷ lệ ca dương tính được phát hiện khi ưu tiên 10% lượt**, không phải tỷ lệ dự đoán đúng toàn bộ.
- Kết quả Bootstrap 95% CI **không** chứng minh mô hình dùng được trong điều trị và không là CI cho dữ liệu bệnh viện khác.
- Chênh lệch DecisionTree–RandomForest rất nhỏ ở quan sát gốc; giải thích khoảng chênh lệch ghép cặp thay vì tuyên bố mô hình thắng tuyệt đối.
- Ở ngưỡng 0,5 có nhiều False Positive; chính sách ngân sách Top 10% là câu hỏi nghiệp vụ khác ngưỡng phân lớp 0,5.

In [ ]:
ci = bootstrap_ci.set_index('metric')
diff=ci.loc['diff_DT_minus_RF_recall_at_10pct']
dt=ci.loc['DecisionTree_recall_at_10pct']
rf=ci.loc['RandomForest_recall_at_10pct']
tie=tie_stats.iloc[0]
summary_lines=[
    'BAO CAO BO SUNG TIEU CHI 7 — TEST, KHONG TOI UU TEST',
    f'Test encounters={n}, positive={int(y.sum())}, patients={num_patients}, follow-up capacity={k}',
    'Locked model=DecisionTree, selected on validation; no refitting',
    f'DT Recall@10% observed={original_top.loc["DecisionTree", "recall_at_10pct"]:.4%}',
    f'DT patient-cluster bootstrap 95% percentile CI=[{dt.q025:.4%}, {dt.q975:.4%}]',
    f'RF patient-cluster bootstrap 95% percentile CI=[{rf.q025:.4%}, {rf.q975:.4%}]',
    f'Paired DT-RF delta Recall@10% bootstrap 95% CI=[{diff.q025:+.4%}, {diff.q975:+.4%}]',
    f'DT top-10 cutoff tie: {int(tie_boundary.iloc[0].n_equal_cutoff)} tied, {int(tie_boundary.iloc[0].slots_from_tie)} slots',
    f'DT randomized tie break: 95% simulation quantiles in hits=[{tie.random_tie_q025_hits:.1f}, {tie.random_tie_q975_hits:.1f}]',
    'Cost scenarios use hypothetical FN:FP ratios 2:1, 5:1, 10:1; not real hospital costs',
    'CI conditionally quantifies uncertainty over Test patient composition, not retraining or shift',
    'Do not export patient-level identifiers / priority lists',
]
(OUT/'evaluation_summary_for_report.txt').write_text('\n'.join(summary_lines)+'\n',encoding='utf-8')
log=pd.DataFrame([
    {'code':'D09-01','decision':'Lock DecisionTree and 10% from validation','reason':'Prevent test-based model selection','impact':'Top-10 comparison uses fixed capacity'},
    {'code':'D09-02','decision':'Bootstrap resample patient clusters','reason':'One patient has multiple encounters','impact':f'{BOOTSTRAP_REPS} replicates, 95% CI'},
    {'code':'D09-03','decision':'Analyze tied scores at Top-10 cutoff','reason':'Stable ranking among tied probabilities depends on row order','impact':f'{TIE_SIMULATIONS} tie-break simulations'},
    {'code':'D09-04','decision':'Cost scenarios C_FN:C_FP=2,5,10 to 1','reason':'Actual clinical cost estimates unavailable','impact':'Scenario sensitivity only, no optimization on Test'},
    {'code':'D09-05','decision':'Export aggregate reports only','reason':'Minimize patient privacy risk','impact':'No ID or patient-level score exported'},
])
log.to_csv(OUT/'decision_log_step9.csv',index=False,encoding='utf-8-sig')
print('\n'.join(summary_lines))
print('\nHoan tat. Output folder:',OUT)